# Extreme Value Theory for Data Scientists

An executable companion to [the full article](article.md). All data are synthetic; the upper tail is a known generalized Pareto distribution, which lets us evaluate extrapolation rather than merely show a fitted curve. This notebook uses a **smaller** seeded experiment for a quick demonstration. Published estimates and eight figures come from python analysis.py.

The problem is not estimating the ordinary event. It is estimating the magnitude, frequency and uncertainty of the rare event.

## The threshold model

Above the fixed threshold u=10, the population has exceedance probability 0.10, shape xi=0.30 and scale beta=3. Its unconditional upper-tail survival is

P(X>x) = 0.10 * [1 + 0.30*(x-10)/3]^(-1/0.30), for x >= 10.

A high quantile p above the threshold can be estimated from the fitted excess distribution and the **unconditional** exceedance fraction. This conversion is essential: the quantile of excesses alone is not the quantile of all events.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import population_survival, population_quantile, run_study, pot_quantile

for probability in (0.9, 0.99, 0.999):
    exact = population_quantile(probability)
    assert np.isclose(float(population_survival(exact)), 1.0 - probability)
print('Known 99.99th percentile:', round(population_quantile(0.9999), 2))

## Extrapolation is sensitive to the assumed tail

The next cell fits a GPD with the correct fixed threshold, along with Gaussian and lognormal whole-distribution benchmarks. The quick run has 5,000 training events: a 1-in-10,000 tail quantity is extrapolated beyond its observed sample frequency, so it may be unstable. The publication uses 40,000 training events and an independent 120,000-event holdout.

In [ ]:
study = run_study(quick=True)
print('Training events:', len(study.training))
print('Exceedances:', study.pot.n_exceed)
print('Fitted GPD shape:', round(study.pot.shape, 3))
for row in study.quantile_rows:
    print('p=', row['probability'], 'truth=', round(float(row['exact_population']), 2), 'POT=', round(float(row['peaks_over_threshold']), 2), 'Gaussian=', round(float(row['all_data_gaussian']), 2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
periods = [float(r['mean_events_between_exceedances_iid']) for r in study.quantile_rows]
for field, label in [('exact_population', 'Known truth'), ('peaks_over_threshold', 'Fitted POT'), ('all_data_gaussian', 'Gaussian')]:
    ax.plot(periods, [float(r[field]) for r in study.quantile_rows], '-o', label=label)
ax.set_xscale('log')
ax.set_xlabel('Marginal event-count return period')
ax.set_ylabel('Estimated magnitude')
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## The hidden role of dependence

Even an exact marginal tail does not determine the frequency of large *blocks* of events. The full experiment builds independent and moving-maximum series with identical one-observation distributions but different clustering. Its reported clusters/exceedance ratios are descriptive finite-threshold quantities, not fitted extremal indices. Do not treat a 1-in-10,000 marginal quantile as a calendar-time forecast without an event rate and dependence model.

In [ ]:
for label in ('iid', 'dependent'):
    selected = next(row for row in study.dependence_rows if row['process'] == label and row['block_size'] == 100)
    print(label, 'clusters/exceedance=', round(float(selected['consecutive_run_clusters_per_exceedance']), 3), 'P(max of 100 > marginal p99)=', round(float(selected['observed_block_exceedance_probability']), 3))
assert len(study.bootstrap_levels) == 18

## Reproducing the publication

Run python analysis.py to generate the full seeded tables and all eight publication PNGs. The CSVs under results/ are the numerical evidence used in the article. Running python analysis.py --quick produces separately suffixed files, leaving the publication figures untouched. The bootstrap interval is conditional on the chosen threshold, GPD family and iid sampling; it is not an automatic guarantee of coverage in a real system.